In [2]:
import os, sys, time
from datetime import datetime
from pyspark.sql import SparkSession, functions as F, types as T

PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB   = os.getenv("PG_DB", "nyc_taxi")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASSWORD", "postgres")
SCHEMA_RAW = os.getenv("PG_SCHEMA_RAW", "raw")

YEARS_ENV   = os.getenv("YEARS", "2015-2025")
SERVICES    = [s.strip() for s in os.getenv("SERVICES","yellow,green").split(",")]
RUN_ID      = os.getenv("RUN_ID", f"run-{int(time.time())}")

def parse_years(r):
    # "2015-2025" o "2019,2020,2021"
    if "-" in r:
        a,b = r.split("-")
        return list(range(int(a), int(b)+1))
    return [int(x) for x in r.split(",")]

YEARS = parse_years(YEARS_ENV)

JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
JDBC_OPTS = {
    "url": JDBC_URL,
    "driver": "org.postgresql.Driver",
    "user": PG_USER,
    "password": PG_PASS,
}

spark = (
    SparkSession.builder
    .appName("ingesta_raw_nyc_taxi")
    .getOrCreate()
)
spark

spark.conf.get("spark.driver.memory"), spark.conf.get("spark.executor.memory")



('6g', '6g')

In [3]:
def common_casts(df, service):
    # Normaliza timestamps y tipos comunes según servicio
    if service == "yellow":
        pickup_col = "tpep_pickup_datetime"
        dropoff_col = "tpep_dropoff_datetime"
    else:
        pickup_col = "lpep_pickup_datetime"
        dropoff_col = "lpep_dropoff_datetime"

    # columnas que suelen estar presentes
    cols_float = [c for c in ["fare_amount","extra","mta_tax","tip_amount","tolls_amount",
                              "improvement_surcharge","total_amount","congestion_surcharge"]
                  if c in df.columns]
    for c in cols_float:
        df = df.withColumn(c, F.col(c).cast("double"))

    for c in [pickup_col, dropoff_col]:
        if c in df.columns:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # agrega metadatos
    df = (df
          .withColumn("ingested_at_utc", F.current_timestamp())
          .withColumn("run_id", F.lit(RUN_ID))
         )
    return df, pickup_col, dropoff_col

def write_df(df, table, mode="append"):
    (df.write.format("jdbc")
       .options(**JDBC_OPTS)
       .option("dbtable", table)
       .mode(mode)
       .save())


In [4]:
import pandas as pd

zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# 1) Descargar con pandas (soporta https)
zones_pdf = pd.read_csv(zones_url)

# 2) Subir a Spark
zones = spark.createDataFrame(zones_pdf)

# 3) Tipos + metadatos
zones = (zones
         .withColumn("LocationID", F.col("LocationID").cast("int"))
         .withColumn("ingested_at_utc", F.current_timestamp())
         .withColumn("run_id", F.lit(RUN_ID)))

# 4) Guardar en Postgres
write_df(zones, f"{SCHEMA_RAW}.taxi_zone_lookup", mode="overwrite")
print("✅ raw.taxi_zone_lookup")


✅ raw.taxi_zone_lookup


In [14]:
# --- Configuración: elige modo selectivo o completo ---
SELECTIVE_MODE = True   # True = solo los meses indicados abajo; False = recorre YEARS x 1..12

# Lista de (service, year, month) a procesar cuando SELECTIVE_MODE=True
SELECTIVE_MONTHS = [
    ("green",  2015, 1),
    ("yellow", 2015, 1),
]

# --- Implementación: descarga temporal, ingesta y limpieza ---
import requests, os, time, traceback, sys, subprocess
from io import BytesIO
import pyarrow.parquet as pq
import pandas as pd

data_tmp = "/home/jovyan/work/tmp_parquet"
os.makedirs(data_tmp, exist_ok=True)

# --- Helpers para conexión a Postgres (para pre-DELETE) ---
try:
    import psycopg2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "psycopg2-binary"], check=True)
    import psycopg2

PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "nyc_taxi")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASSWORD", "postgres")

def pg_exec(sql, params=None):
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS
    )
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params or [])
    finally:
        conn.close()

def delete_partition(service, y, m):
    tbl = target_table(service)
    log(f"Pre-DELETE → {tbl} (service={service}, year={y}, month={m})", "INFO")
    pg_exec(f"DELETE FROM {tbl} WHERE service=%s AND year=%s AND month=%s;", [service, y, m])
    log("Pre-DELETE completado", "OK")


def trip_url(service, year, month):
    mm = f"{month:02d}"
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{service}_tripdata_{year}-{mm}.parquet"

def target_table(service):
    return f"{SCHEMA_RAW}.{service}_taxi_trip"

def log(msg, level="INFO"):
    """Log con formato uniforme."""
    ts = time.strftime("%H:%M:%S")
    icons = {"INFO": "ℹ️", "OK": "✅", "WARN": "⚠️", "ERROR": "❌"}
    print(f"[{ts}] {icons.get(level, '')} {msg}")


def ingest_one_month(service, y, m):
    """Descarga un parquet (temporal), lo ingesta idempotente (pre-DELETE) y lo borra."""
    url = trip_url(service, y, m)
    tmp_path = f"{data_tmp}/{service}_{y}-{m:02d}.parquet"
    log(f"Iniciando {service.upper()} {y}-{m:02d}", "INFO")
    start = time.time()
    try:
        # 1️⃣ Descarga
        log(f"Descargando {url} ...", "INFO")
        r = requests.get(url, timeout=180)
        if r.status_code != 200:
            log(f"No encontrado ({r.status_code})", "WARN")
            return False

        with open(tmp_path, "wb") as f:
            f.write(r.content)
        log(f"Archivo descargado: {round(os.path.getsize(tmp_path)/1024/1024,2)} MB", "OK")

        # 2️⃣ Lectura con Spark
        log("Leyendo Parquet local con Spark...", "INFO")
        df = spark.read.parquet(tmp_path)
        log("Schema de origen:", "INFO")
        df.printSchema()

        # Normalización de tipos + metadatos
        df, pickup_col, dropoff_col = common_casts(df, service)
        df = (df
              .withColumn("service", F.lit(service))
              .withColumn("year", F.lit(int(y)))
              .withColumn("month", F.lit(int(m))))

        sample_row = df.limit(1).toPandas().to_dict(orient="records")
        log(f"Fila de muestra: {sample_row}", "INFO")

        # 3️⃣ Idempotencia: eliminar antes de escribir
        tbl = target_table(service)
        delete_partition(service, y, m)

        # 4️⃣ Escritura
        log(f"Escribiendo en Postgres → {tbl}", "INFO")
        write_df(df, tbl, mode="append")
        log(f"{service.upper()} {y}-{m:02d} cargado correctamente en {tbl}", "OK")
        return True

    except Exception as e:
        log(f"Error en {service} {y}-{m:02d}: {e}", "ERROR")
        traceback.print_exc(limit=1)
        return False

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
            log(f"Archivo temporal eliminado: {tmp_path}", "INFO")
        elapsed = round(time.time() - start, 2)
        log(f"Tiempo total {service} {y}-{m:02d}: {elapsed}s", "INFO")
        print("-"*80)


# --- Ejecución según el modo ---
if SELECTIVE_MODE:
    log("===> Modo SELECTIVO", "INFO")
    for service, y, m in SELECTIVE_MONTHS:
        _ = ingest_one_month(service, y, m)
else:
    log("===> Modo COMPLETO", "INFO")
    for service in SERVICES:
        log(f"Servicio: {service}", "INFO")
        for y in YEARS:
            for m in range(1, 13):
                _ = ingest_one_month(service, y, m)

log("🏁 Ingesta terminada (según modo seleccionado)", "OK")


[19:59:01] ℹ️ ===> Modo SELECTIVO
[19:59:01] ℹ️ Iniciando GREEN 2015-01
[19:59:01] ℹ️ Descargando https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2015-01.parquet ...
[19:59:13] ✅ Archivo descargado: 21.68 MB
[19:59:13] ℹ️ Leyendo Parquet local con Spark...
[19:59:13] ℹ️ Schema de origen:
root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surcha

In [15]:
for service in SERVICES:
    tbl = target_table(service)
    q = f"(select count(*) as n, min(year) as y0, max(year) as y1 from {tbl}) as t"
    df = (spark.read.format("jdbc")
          .options(**JDBC_OPTS)
          .option("dbtable", q)
          .load())
    df.show(truncate=False)


+--------+----+----+
|n       |y0  |y1  |
+--------+----+----+
|12741035|2015|2015|
+--------+----+----+

+-------+----+----+
|n      |y0  |y1  |
+-------+----+----+
|1508493|2015|2015|
+-------+----+----+



In [16]:
for service in SERVICES:
    tbl = target_table(service)
    q = f"(select service, year, month, count(*) as n from {tbl} group by 1,2,3 order by 2,3 limit 24) t"
    df = (spark.read.format("jdbc")
          .options(**JDBC_OPTS)
          .option("dbtable", q)
          .load())
    print(f"### {tbl}")
    df.orderBy("year","month").show(50, truncate=False)


### raw.yellow_taxi_trip
+-------+----+-----+--------+
|service|year|month|n       |
+-------+----+-----+--------+
|yellow |2015|1    |12741035|
+-------+----+-----+--------+

### raw.green_taxi_trip
+-------+----+-----+-------+
|service|year|month|n      |
+-------+----+-----+-------+
|green  |2015|1    |1508493|
+-------+----+-----+-------+



In [30]:
# --- Resumen por servicio/año/mes
q_counts = """
(SELECT service, year, month, COUNT(*) AS n
   FROM analytics.obt_trips
  GROUP BY 1,2,3
  ORDER BY 1,2,3) t
"""
df_counts = (spark.read.format("jdbc")
             .options(**JDBC_OPTS)
             .option("dbtable", q_counts)
             .load())
print("### analytics.obt_trips — conteos por partición")
df_counts.show(100, truncate=False)

# --- Totales globales
q_tot = "(SELECT COUNT(*) AS rows, SUM(total_amount) AS total_amount FROM analytics.obt_trips) t"
df_tot = (spark.read.format("jdbc")
          .options(**JDBC_OPTS)
          .option("dbtable", q_tot)
          .load())
print("### Totales globales")
df_tot.show(truncate=False)


### analytics.obt_trips — conteos por partición
+-------+----+-----+--------+
|service|year|month|n       |
+-------+----+-----+--------+
|green  |2015|1    |1506259 |
|yellow |2015|1    |12736668|
+-------+----+-----+--------+

### Totales globales
+--------+------------------+
|rows    |total_amount      |
+--------+------------------+
|14242927|2.14852465449131E8|
+--------+------------------+



In [31]:
# --- Top zonas de pickup por viajes y por facturación (ajusta año/mes si quieres)
YEAR, MONTH = 2015, 1

q_top_trips = f"""
(SELECT pu_borough, pu_zone, COUNT(*) AS trips
   FROM analytics.obt_trips
  WHERE year={YEAR} AND month={MONTH}
  GROUP BY 1,2
  ORDER BY trips DESC
  LIMIT 10) t
"""
q_top_amount = f"""
(SELECT pu_borough, pu_zone, SUM(total_amount) AS total_usd
   FROM analytics.obt_trips
  WHERE year={YEAR} AND month={MONTH}
  GROUP BY 1,2
  ORDER BY total_usd DESC
  LIMIT 10) t
"""

df_top_trips = (spark.read.format("jdbc").options(**JDBC_OPTS).option("dbtable", q_top_trips).load())
df_top_amount = (spark.read.format("jdbc").options(**JDBC_OPTS).option("dbtable", q_top_amount).load())

print(f"### Top 10 zonas por viajes — {YEAR}-{MONTH:02d}")
df_top_trips.show(truncate=False)
print(f"### Top 10 zonas por facturación — {YEAR}-{MONTH:02d}")
df_top_amount.show(truncate=False)

# --- Chequeos de calidad
q_nulls = f"""
(SELECT
    SUM(CASE WHEN pickup_ts IS NULL THEN 1 ELSE 0 END) AS null_pickup,
    SUM(CASE WHEN dropoff_ts IS NULL THEN 1 ELSE 0 END) AS null_dropoff,
    SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) AS null_pu_loc,
    SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END) AS null_do_loc,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS null_total
  FROM analytics.obt_trips
 WHERE year={YEAR} AND month={MONTH}) t
"""
df_nulls = (spark.read.format("jdbc").options(**JDBC_OPTS).option("dbtable", q_nulls).load())
print("### Nulos clave (pickup/dropoff/zona/total)")
df_nulls.show(truncate=False)

# --- Muestra de filas (para evidencias)
q_sample = f"""
(SELECT service, year, month, vendor_id, pickup_ts, dropoff_ts,
        pu_borough, pu_zone, do_borough, do_zone,
        total_amount, airport_fee, cbd_congestion_fee, trip_type
   FROM analytics.obt_trips
  WHERE year={YEAR} AND month={MONTH}
  LIMIT 5) t
"""
df_sample = (spark.read.format("jdbc").options(**JDBC_OPTS).option("dbtable", q_sample).load())
print("### Muestra de filas")
df_sample.show(truncate=False)


### Top 10 zonas por viajes — 2015-01
+----------+----------------------------+------+
|pu_borough|pu_zone                     |trips |
+----------+----------------------------+------+
|Manhattan |Upper East Side South       |467013|
|Manhattan |Midtown Center              |446282|
|Manhattan |Upper East Side North       |445117|
|Manhattan |East Village                |437630|
|Manhattan |Times Sq/Theatre District   |429122|
|Manhattan |Union Sq                    |423061|
|Manhattan |Murray Hill                 |411186|
|Manhattan |Midtown East                |410487|
|Manhattan |Clinton East                |400812|
|Manhattan |Penn Station/Madison Sq West|392878|
+----------+----------------------------+------+

### Top 10 zonas por facturación — 2015-01
+----------+-------------------------+--------------------+
|pu_borough|pu_zone                  |total_usd           |
+----------+-------------------------+--------------------+
|Queens    |JFK Airport              |1.283756118998

In [33]:
# === Parámetros ===
YEAR   = 2015     # cambia si quieres
MONTH  = 1        # cambia si quieres
SERVICES = ["yellow","green"]  # o ["yellow"] / ["green"]

from pyspark.sql import functions as F

svc_list = ",".join([f"'{s}'" for s in SERVICES])
base_sql = f"""
(SELECT *
   FROM analytics.obt_trips
  WHERE year={YEAR} AND month={MONTH}
    AND service IN ({svc_list})
)
"""

df = (spark.read.format("jdbc")
      .options(**JDBC_OPTS)
      .option("dbtable", base_sql)
      .load())

print("### Muestra")
df.select("service","year","month","vendor_id","pickup_ts","dropoff_ts",
          "trip_distance","total_amount","passenger_count","ratecode_id","payment_type").show(5, truncate=False)

# --- Derivadas útiles ---
df2 = (
    df
    .withColumn("duration_min", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long"))/60.0)
    .withColumn("duration_hr",  F.col("duration_min")/60.0)
    .withColumn("mph", F.when(F.col("duration_hr")>0, F.col("trip_distance")/F.col("duration_hr")))
)

# --- Reglas de calidad (puedes ajustar umbrales si tu profesor lo pide distinto) ---
bad = {
  "null_timestamps"     : (F.col("pickup_ts").isNull() | F.col("dropoff_ts").isNull()),
  "inverted_times"      : (F.col("pickup_ts") > F.col("dropoff_ts")),
  "neg_distance"        : (F.col("trip_distance") < 0),
  "huge_distance"       : (F.col("trip_distance") > 200),   # 200 millas intra-NYC es sospechoso
  "neg_amount"          : (F.col("total_amount") < 0),
  "huge_amount"         : (F.col("total_amount") > 1000),   # > $1000 muy raro
  "pc_out_of_range"     : (F.col("passenger_count").isNotNull() & ~F.col("passenger_count").between(0,8)),
  "ratecode_out"        : (F.col("ratecode_id").isNotNull() & ~F.col("ratecode_id").isin(1,2,3,4,5,6,99)),
  "payment_out"         : (F.col("payment_type").isNotNull() & ~F.col("payment_type").isin(0,1,2,3,4,5,6)),
  "zero_or_neg_duration": (F.col("duration_min") <= 0),
  "ultra_long_duration" : (F.col("duration_min") > 600),     # >10 horas
  "mph_too_fast"        : (F.col("mph") > 100),              # >100 mph sostenido improbable
  "mph_too_slow"        : (F.col("mph") < 1) & (F.col("trip_distance") > 0.5)  # muy lento para >0.5mi
}

# --- Resumen por servicio ---
aggs = [F.sum(F.when(cond, F.lit(1)).otherwise(F.lit(0))).alias(name) for name, cond in bad.items()]
summary = (df2.groupBy("service")
           .agg(F.count(F.lit(1)).alias("rows"), *aggs)
           .orderBy("service"))
print("### Resumen de anomalías por servicio")
summary.show(truncate=False)

# --- Totales globales (del filtro YEAR/MONTH/SERVICES) ---
summary_tot = df2.agg(F.count(F.lit(1)).alias("rows_total"), *aggs)
print("### Totales de anomalías (periodo seleccionado)")
summary_tot.show(truncate=False)

# --- Percentiles rápidos para contexto (distancia, total, duración, mph) ---
def pct_query(col):
    return (df2
            .select(col)
            .summary("count","min","25%","50%","75%","95%","99%","max"))
print("### Percentiles trip_distance"); pct_query("trip_distance").show(truncate=False)
print("### Percentiles total_amount");  pct_query("total_amount").show(truncate=False)
print("### Percentiles duration_min"); pct_query("duration_min").show(truncate=False)
print("### Percentiles mph");          pct_query("mph").show(truncate=False)

# --- Muestras de filas problemáticas (hasta 5 por tipo) ---
cols_show = ["service","pickup_ts","dropoff_ts","trip_distance","total_amount",
             "passenger_count","ratecode_id","payment_type","duration_min","mph","pu_zone","do_zone"]

def show_examples(label, condition):
    print(f"\n### Ejemplos: {label}")
    (df2
     .filter(condition)
     .select(*cols_show)
     .orderBy(F.col("pickup_ts").asc_nulls_last())
     .limit(5)
     .show(truncate=False))

for label, condition in bad.items():
    show_examples(label, condition)


### Muestra
+-------+----+-----+---------+-------------------+-------------------+-------------+------------+---------------+-----------+------------+
|service|year|month|vendor_id|pickup_ts          |dropoff_ts         |trip_distance|total_amount|passenger_count|ratecode_id|payment_type|
+-------+----+-----+---------+-------------------+-------------------+-------------+------------+---------------+-----------+------------+
|yellow |2015|1    |1        |2015-01-01 00:11:33|2015-01-01 00:16:48|1.0          |8.4         |1              |1          |1           |
|yellow |2015|1    |1        |2015-01-01 00:18:24|2015-01-01 00:24:20|0.9          |7.3         |1              |1          |3           |
|yellow |2015|1    |1        |2015-01-01 00:26:19|2015-01-01 00:41:06|3.5          |17.4        |1              |1          |1           |
|yellow |2015|1    |1        |2015-01-01 00:45:26|2015-01-01 00:53:20|2.1          |11.87       |1              |1          |1           |
|yellow |2015|1